# 🔌 Utility Asset Detector

**One-click setup for detecting T&D utility infrastructure.**

This notebook runs on Google Colab with a free GPU. Detects:
- Utility poles (wood, concrete, steel)
- Transmission towers
- Insulators, transformers, cross-arms
- Damage: cracks, rust, woodpecker holes, lean, etc.

**Instructions:**
1. Click `Runtime → Run all` (or Ctrl+F9)
2. Wait ~3 minutes for installation
3. Use the Gradio interface that appears

---

In [ ]:
#@title 1️⃣ Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(f"\n✅ PyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title 2️⃣ Install Dependencies (~2 min)
!pip install -q gradio pillow opencv-python pyyaml

# Clone and install DART
!git clone -q https://github.com/mkturkcan/DART.git
%cd DART
!pip install -q -e .
%cd ..

# Clone utility-asset-detector
!git clone -q https://github.com/menonpg/utility-asset-detector.git
%cd utility-asset-detector

print("\n✅ Installation complete!")

In [ ]:
#@title 3️⃣ Quick Test (Downloads SAM3 checkpoint ~1.7GB on first run)
# This cell runs a quick detection to trigger the auto-download
# The checkpoint downloads automatically from HuggingFace

import sys
sys.path.insert(0, ".")

from PIL import Image
import requests
from io import BytesIO

# Download a test image
print("Downloading test image...")
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/Wooden_utility_pole.jpg/400px-Wooden_utility_pole.jpg"
response = requests.get(url)
test_image = Image.open(BytesIO(response.content))
test_image.save("test_pole.jpg")
print("✅ Test image ready")

# Initialize detector (this triggers SAM3 checkpoint download)
print("\nInitializing detector (downloading SAM3 checkpoint on first run)...")
print("⏳ This may take 2-3 minutes...\n")

from src.detector import UtilityAssetDetector
detector = UtilityAssetDetector(
    config="configs/assets.yaml",
    device="cuda"
)

# Run quick test
result = detector.detect("test_pole.jpg")
print(f"\n✅ Detector ready!")
print(f"   Found {result.total_structures} structures, {result.total_components} components")

In [ ]:
#@title 4️⃣ Launch Web UI 🚀
from app import create_ui

app = create_ui()
app.launch(
    share=True,  # Creates public URL
    debug=True,
)

---

## 📝 Alternative: Run Detection Directly

If you prefer code over the UI:

In [ ]:
#@title Direct Detection on Your Image
from google.colab import files
from src.detector import UtilityAssetDetector
from PIL import Image
import matplotlib.pyplot as plt

# Upload your image
print("Upload an image:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    
    # Initialize detector
    detector = UtilityAssetDetector(
        config="configs/assets.yaml",
        device="cuda"
    )
    
    # Run detection
    result = detector.detect(filename)
    
    # Print results
    print(f"\n📊 Results:")
    print(f"Structures found: {result.total_structures}")
    print(f"Components found: {result.total_components}")
    print(f"Priority: {result.priority}")
    print()
    
    for struct in result.structures:
        print(f"📍 {struct.type} ({struct.confidence:.0%})")
        print(f"   Status: {struct.condition.status}")
        if struct.condition.issues:
            print(f"   Issues: {', '.join(struct.condition.issues)}")
        for comp in struct.components:
            print(f"   - {comp.type}: {comp.condition.status}")

In [ ]:
#@title Save JSON Results
# Save detection results to file
with open("detection_result.json", "w") as f:
    f.write(result.to_json())

print("✅ Saved to detection_result.json")
print(result.to_json())

# Download the file
from google.colab import files
files.download("detection_result.json")